# LoRA fine-tune of Llama3.2-3B for PikaRAG

LoRA fine-tune of Llama3.2-3B on PikaRAG's generated training pairs
(`data/finetune/train.jsonl`), run once on a free-tier Colab T4 GPU.
Training only -- serving stays local via Ollama, see
`docs/finetuned-model-serving.md` for the merge/convert/quantize/deploy
steps after this notebook produces an adapter.

The base model choice (`meta-llama/Llama-3.2-3B-Instruct`) mirrors the
live RAG path's `llama3.2:3b`, for an apples-to-apples comparison
against `/ask`'s RAG answers via `scripts/run_eval.py --model rag`
vs. `--model finetuned` (see
`docs/superpowers/specs/2026-09-17-finetune-vs-rag-design.md`).

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate datasets

In [ ]:
from google.colab import files
uploaded = files.upload()  # select data/finetune/train.jsonl

In [ ]:
import json
from datasets import Dataset

pairs = [json.loads(line) for line in open("train.jsonl")]

def to_chat_text(pair):
    return (
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{pair['question']}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{pair['answer']}<|eot_id|>"
    )

dataset = Dataset.from_list([{"text": to_chat_text(p)} for p in pairs])
print(dataset)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=256, padding="max_length")

tokenized = dataset.map(tokenize, remove_columns=["text"])

training_args = TrainingArguments(
    output_dir="./pikarag-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()

In [ ]:
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./pikarag-finetuned-merged", safe_serialization=True)
tokenizer.save_pretrained("./pikarag-finetuned-merged")

In [ ]:
import shutil
shutil.make_archive("pikarag-finetuned-merged", "zip", "pikarag-finetuned-merged")
files.download("pikarag-finetuned-merged.zip")

Next: follow `docs/finetuned-model-serving.md` on the machine serving
Ollama for this project to convert the downloaded merged model to a
quantized GGUF and register it as `pikarag-finetuned`.